# Data Engineering Project Plan
## End-to-End Medallion Architecture Implementation

---

## Project Overview
Implement a complete data pipeline following the Medallion Architecture (Bronze → Silver → Gold) with CDC implementation and analytics.

---

## Phase 1: Data Generation & Bronze Layer
### Objective: Create sample datasets with realistic data quality issues

#### Tables to Create:
1. **`customers`** - Customer master data
   - Columns: `customer_id`, `name`, `email`, `city`, `state`, `signup_date`, `phone`
   - Data Quality: Include NULL emails, duplicate customer_ids
   
2. **`products`** - Product catalog
   - Columns: `product_id`, `product_name`, `category`, `price`, `supplier_id`
   - Data Quality: Include NULL prices, duplicate product_ids
   
3. **`sales`** - Sales transactions
   - Columns: `sale_id`, `customer_id`, `product_id`, `quantity`, `sale_amount`, `sale_date`, `region`
   - Data Quality: Include NULL quantities, orphan customer/product references
   
4. **`suppliers`** - Supplier information
   - Columns: `supplier_id`, `supplier_name`, `contact_email`, `country`
   - Data Quality: Include NULL contact info

#### Deliverables:
- Python/SQL scripts to generate sample data (~1000 records per table)
- Save to Bronze layer: `dev.bronze.customers_raw`, `dev.bronze.products_raw`, `dev.bronze.sales_raw`, `dev.bronze.suppliers_raw`

---

## Phase 2: Ingestion Layer
### Objective: Load data into Bronze layer using scalable ingestion methods

#### Approach Options:
1. **COPY INTO** (simpler, good for batch loads)
   - Export generated data to cloud storage (S3/ADLS/GCS)
   - Use `COPY INTO` to load files into Bronze tables
   - Handle file formats (CSV/JSON/Parquet)

2. **Lakeflow Spark Declarative Pipeline** (advanced, streaming capable)
   - Create DLT pipeline for continuous ingestion
   - Auto Loader for incremental file processing
   - Built-in data quality checks

#### Deliverables:
- Ingestion notebook/pipeline
- Bronze layer tables populated
- Data quality validation queries

---

## Phase 3: CDC Implementation & Silver Layer
### Objective: Apply Change Data Capture with SCD Type 1 and Type 2

#### CDC Implementation:

**Table 1: `customers` → SCD Type 1 (Overwrite changes)**
- Target: `dev.silver.customers_clean`
- Strategy: Latest record wins, no history tracking
- MERGE logic: Update matching records, insert new ones
- Cleanup: Deduplicate, handle NULLs

**Table 2: `products` → SCD Type 2 (Historical tracking)**
- Target: `dev.silver.products_scd2`
- Strategy: Preserve history with versioning
- Additional columns:
  - `effective_date` - When this version became active
  - `end_date` - When this version expired (NULL for current)
  - `is_current` - Boolean flag for current record
  - `version` - Version number
- MERGE logic:
  - Close expired versions (set end_date, is_current=false)
  - Insert new versions for changed records

#### Deliverables:
- SCD Type 1 MERGE statement for customers
- SCD Type 2 MERGE statements for products (2-step process)
- Silver layer tables with cleaned data
- History validation queries

---

## Phase 4: Gold Layer Analytics
### Objective: Build aggregated views and KPIs for business reporting

#### Analytics Requirements:

**KPI 1: Sales Performance Metrics**
- Total revenue by region, product category, time period
- Year-over-year growth rates
- Top 10 customers by revenue
- Window functions: `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`

**KPI 2: Customer Analytics**
- Customer lifetime value (CLV)
- New vs returning customers
- Average order value by customer segment
- Window functions: `LAG()`, `LEAD()`, `FIRST_VALUE()`

**KPI 3: Product Performance**
- Best-selling products by category
- Product price changes over time (using SCD Type 2 history)
- Supplier performance metrics
- Window functions: `SUM() OVER()`, `AVG() OVER()`, `PERCENT_RANK()`

**KPI 4: Trend Analysis**
- Moving averages (7-day, 30-day)
- Running totals
- Cumulative metrics
- Window functions: `SUM() OVER(ORDER BY date ROWS BETWEEN ...)`

#### Complex Joins:
- 4-way join: sales → customers → products → suppliers
- Self-joins for comparison analysis
- Cross joins for date dimension generation

#### Deliverables:
- Gold layer aggregate tables:
  - `dev.gold.sales_summary`
  - `dev.gold.customer_metrics`
  - `dev.gold.product_analytics`
  - `dev.gold.daily_kpis`
- SQL queries demonstrating all window functions
- Dashboard-ready views

---

## Phase 5: Dashboard & Visualization
### Objective: Create interactive dashboards for business users

#### Databricks Dashboard Creation:
1. **Sales Performance Dashboard**
   - Revenue trends by region and category
   - Top customers and products charts
   - Year-over-year comparison visualizations
   - Bar charts, line charts, and pie charts

2. **Customer Analytics Dashboard**
   - Customer lifetime value distribution
   - Customer segmentation analysis
   - Cohort retention analysis
   - Geographic heatmaps

3. **Product Insights Dashboard**
   - Best-selling products table
   - Price history trends (using SCD Type 2 data)
   - Supplier performance metrics
   - Inventory turnover rates

#### Dashboard Features:
- Add filters (date range, region, category)
- Create drill-down capabilities
- Set up auto-refresh schedules
- Share with stakeholders

---

## Phase 6: Genie Intelligence Layer
### Objective: Enable natural language queries with Genie

#### Genie Data Room Setup:
1. **Create Genie Space**
   - Add Gold layer tables to Genie space
   - Include: `dev.gold.sales_summary`, `dev.gold.customer_metrics`, `dev.gold.product_analytics`

2. **Natural Language Queries Examples**
   - "Show me top 10 customers by revenue this year"
   - "What's the average order value by region?"
   - "Which products have the highest profit margins?"
   - "Compare sales performance Q1 vs Q2"
   - "Show me customers who haven't purchased in 90 days"

3. **Genie Benefits**
   - Self-service analytics for business users
   - No SQL knowledge required
   - Conversational interface for data exploration
   - Automatic visualization generation

#### Deliverables:
- Genie space with Gold tables
- Documentation of common business questions
- Training guide for business users

---

## Phase 7: Orchestration & Testing
### Objective: Automate and validate the pipeline

#### Tasks:
1. **Workflow Creation**
   - Schedule Bronze ingestion (daily/hourly)
   - Schedule Silver CDC processing
   - Schedule Gold aggregations
   - Schedule Dashboard refresh
   
2. **Data Quality Tests**
   - Row count validations
   - NULL check assertions
   - Referential integrity checks
   - SCD Type 2 history verification
   
3. **Performance Optimization**
   - Add appropriate partitioning
   - Create Z-ORDER indexes
   - Optimize MERGE operations

---

## Success Criteria

### Core Pipeline:
✅ All 4 Bronze tables loaded with quality issues
✅ COPY INTO or pipeline successfully ingests data
✅ SCD Type 1 correctly overwrites customer changes
✅ SCD Type 2 preserves product history with proper versioning
✅ Gold layer contains 4+ KPI views
✅ Queries demonstrate 5+ different window functions
✅ All joins working correctly across layers
✅ **Databricks Dashboard created with 5+ visualizations**
✅ **Genie Data Room set up with Gold tables**
✅ **Natural language queries working in Genie**

### Data Governance & Security:
✅ **Row-level security implemented for sales and customers**
✅ **Column-level security with dynamic data masking for PII**
✅ **User groups and role-based permissions configured**
✅ **Unity Catalog tags applied (pii_level, data_classification)**
✅ **Audit logging queries documented and tested**
✅ **Data lineage fully documented**

### SDLC & Automation:
✅ **DABs project structure with databricks.yml**
✅ **Multi-environment configuration (dev/uat/prod)**
✅ **CI/CD pipeline integrated with Git**
✅ **Successfully deployed to all three environments**
✅ **Automated workflows scheduled for production**
✅ **Monitoring and alerting configured**
✅ **Operational runbook and rollback procedures documented**

---

## Implementation Phases (3 Weeks)

### Week 1: Core Data Pipeline
1. Create Bronze data generation scripts
2. Set up ingestion pipeline (COPY INTO or Lakeflow)
3. Implement CDC transformations (SCD Type 1 & Type 2)
4. Build Gold analytics layer with window functions

### Week 2: Visualization & Governance
5. Create Databricks Dashboard with visualizations
6. Set up Genie Data Room for natural language queries
7. Implement Row-Level Security (RLS)
8. Implement Column-Level Security (CLS) and data masking
9. Configure Unity Catalog governance (tags, comments, permissions)
10. Set up audit logging and compliance reporting

### Week 3: SDLC & Production
11. Initialize DABs project and configure multi-environment setup
12. Define job resources and pipeline configurations
13. Deploy to dev → test → deploy to UAT → test
14. User acceptance testing and bug fixes
15. Production deployment with monitoring and alerting
16. Create operational runbooks and documentation

---

## Project Timeline Summary
**Duration:** 3 weeks (90 hours total)
**Weekly Effort:** 30 hours per week (~6 hours per day)
**Environments:** DEV → UAT → PROD
**Delivery:** Production-ready enterprise data platform with governance and CI/CD

## Implementation Timeline - 3 Week Sprint (90 Hours Total)
**30 hours per week | ~6 hours per day over 5 days**

---

# Week 1: Foundation & Core Pipeline (30 hours)

## Day 1-2: Setup & Bronze Layer (12 hours)
### Monday (6 hours)
- **Hours 1-2**: Create multi-environment catalog structure
  - `dev.bronze`, `dev.silver`, `dev.gold`
  - `uat.bronze`, `uat.silver`, `uat.gold`
  - `prod.bronze`, `prod.silver`, `prod.gold`
- **Hours 3-4**: Generate sample data scripts
  - Customers data (1000 rows, with nulls/duplicates)
  - Products data (500 rows, with nulls/duplicates)
  - Sales data (5000 rows, with nulls)
  - Suppliers data (100 rows)
- **Hours 5-6**: Load data to Bronze tables in dev environment
  - Run validation queries to confirm nulls and duplicates exist

### Tuesday (6 hours)
- **Hours 1-3**: Implement ingestion layer
  - Choose between COPY INTO or Lakeflow Pipeline
  - Export Bronze data to cloud storage (if using COPY INTO)
  - Create ingestion scripts/pipeline
- **Hours 4-6**: Build SCD Type 1 for customers
  - Create `dev.silver.customers_clean` table
  - Write MERGE logic with deduplication
  - Test and validate customer data in Silver layer

## Day 3: Silver - SCD Type 2 (6 hours)
### Wednesday (6 hours)
- **Hours 1-2**: Create products_scd2 table structure
  - Add `effective_date`, `end_date`, `is_current`, `version` columns
- **Hours 3-5**: Implement SCD Type 2 MERGE logic
  - Close expired versions (set end_date, is_current=false)
  - Insert new versions for changed records
- **Hour 6**: Test with product price changes and verify history preservation

## Day 4-5: Gold Layer & Initial Analytics (12 hours)
### Thursday (6 hours)
- **Hours 1-3**: Create Gold aggregate tables
  - `dev.gold.sales_summary`
  - `dev.gold.customer_metrics`
  - `dev.gold.product_analytics`
  - `dev.gold.daily_kpis`
- **Hours 4-6**: Build KPI queries with window functions
  - Implement ROW_NUMBER, RANK, DENSE_RANK
  - Add LAG, LEAD, FIRST_VALUE
  - Create moving averages and running totals

### Friday (6 hours)
- **Hours 1-3**: Implement 4-way joins and complex analytics
  - Join sales → customers → products → suppliers
  - Create self-joins for comparison analysis
- **Hours 4-6**: Initial testing and validation
  - Verify data quality across all layers
  - Document any issues found

---

# Week 2: Visualization, Intelligence & Data Governance (30 hours)

## Day 6-7: Dashboards & Genie Setup (12 hours)
### Monday (6 hours)
- **Hours 1-4**: Create Databricks Dashboard
  - Sales Performance charts (revenue trends, regional analysis)
  - Customer Analytics visualizations (CLV, segmentation)
  - Product Insights (best-sellers, price history)
- **Hours 5-6**: Add dashboard features (filters, drill-downs, auto-refresh)

### Tuesday (6 hours)
- **Hours 1-3**: Set up Genie Data Room
  - Add Gold layer tables to Genie space
  - Configure natural language query interface
- **Hours 4-6**: Test and document common business questions
  - Create sample queries for business users
  - Build training documentation

## Day 8-10: Data Governance & Security (18 hours)
### Wednesday (6 hours)
- **Hours 1-2**: Unity Catalog governance setup
  - Document data lineage
  - Add table/column comments and tags
- **Hours 3-4**: Implement Row-Level Security (RLS)
  - Create row filters using `CASE WHEN current_user() = ...`
  - Apply filters to sensitive tables (customers, sales)
  - Test with different users/roles
- **Hours 5-6**: Implement Column-Level Security (CLS)
  - Create column masks for PII data (email, phone)
  - Use `CASE WHEN is_account_group_member()` for role-based access

### Thursday (6 hours)
- **Hours 1-3**: Implement Dynamic Data Masking
  - Mask email addresses: `***@domain.com`
  - Mask phone numbers: `***-***-1234`
  - Partial SSN masking: `***-**-1234`
  - Credit card masking: `****-****-****-1234`
- **Hours 4-6**: Create secured views with masking logic
  - `dev.gold.customers_secure_view`
  - `dev.gold.sales_secure_view`
  - Test masking with different user groups

### Friday (6 hours)
- **Hours 1-3**: Fine-grained access control setup
  - Create user groups: `data_engineers`, `analysts`, `executives`
  - Grant appropriate permissions (SELECT, MODIFY, ALL PRIVILEGES)
  - Implement least privilege access model
- **Hours 4-6**: Governance documentation & audit setup
  - Document all security policies
  - Set up audit logging for sensitive data access
  - Create compliance reports

---

# Week 3: SDLC with DABs & Production Deployment (30 hours)

## Day 11-12: Declarative Automation Bundles (DABs) Setup (12 hours)
### Monday (6 hours)
- **Hours 1-2**: Initialize DAB project structure
  - Run `databricks bundle init`
  - Review generated folder structure (src/, resources/, databricks.yml)
- **Hours 3-4**: Configure databricks.yml for multi-environment
  - Define `dev`, `uat`, `prod` targets
  - Set environment-specific variables (catalog names, clusters, permissions)
  - Configure resource paths and deployment settings
- **Hours 5-6**: Organize pipeline code into bundle structure
  - Move notebooks/scripts to `src/` folder
  - Create reusable modules and libraries

### Tuesday (6 hours)
- **Hours 1-3**: Define bundle resources
  - Add workflow/job definitions in `resources/jobs/`
  - Add pipeline definitions in `resources/pipelines/`
  - Configure cluster policies and compute resources
- **Hours 4-6**: Set up CI/CD integration
  - Configure Git integration
  - Create deployment workflows
  - Document bundle deployment process

## Day 13-14: Multi-Environment Deployment & Testing (12 hours)
### Wednesday (6 hours)
- **Hours 1-2**: Deploy to DEV environment
  - Run `databricks bundle validate -t dev`
  - Run `databricks bundle deploy -t dev`
  - Verify all resources created successfully
- **Hours 3-4**: Test end-to-end pipeline in DEV
  - Run Bronze → Silver → Gold pipeline
  - Validate data quality and transformations
  - Test security policies and masking
- **Hours 5-6**: Deploy to UAT environment
  - Run `databricks bundle deploy -t uat`
  - Configure UAT-specific settings
  - Load UAT test data

### Thursday (6 hours)
- **Hours 1-3**: UAT testing & validation
  - Business user acceptance testing
  - Dashboard and Genie testing in UAT
  - Security and governance validation
  - Performance testing
- **Hours 4-6**: Bug fixes and refinements
  - Address issues found during UAT
  - Update documentation
  - Prepare production deployment checklist

## Day 15: Production Deployment & Documentation (6 hours)
### Friday (6 hours)
- **Hours 1-2**: Production deployment
  - Run `databricks bundle validate -t prod`
  - Run `databricks bundle deploy -t prod`
  - Verify production resources
- **Hours 3-4**: Configure production workflows
  - Schedule jobs for Bronze ingestion (daily/hourly)
  - Schedule Silver CDC processing
  - Schedule Gold aggregations
  - Set up monitoring and alerting
- **Hours 5-6**: Final documentation & handoff
  - Create operational runbook
  - Document troubleshooting procedures
  - Prepare training materials
  - Project retrospective and lessons learned

---

## Total Timeline Summary

| Week | Focus Areas | Hours |
|------|-------------|-------|
| Week 1 | Foundation, Bronze, Silver, Gold Core | 30 |
| Week 2 | Dashboards, Genie, Data Governance & Security | 30 |
| Week 3 | DABs, SDLC, Multi-Env Deployment | 30 |
| **Total** | **Complete End-to-End Data Platform** | **90** |

---

## Key Deliverables by Week

### Week 1 Deliverables:
✅ Multi-environment catalog structure (dev/uat/prod)
✅ Bronze layer with 4 tables
✅ Silver layer with SCD Type 1 & Type 2
✅ Gold layer with analytics tables
✅ Ingestion pipeline operational

### Week 2 Deliverables:
✅ Interactive Databricks Dashboard
✅ Genie Data Room for natural language queries
✅ Row-level security implemented
✅ Column-level security implemented
✅ Dynamic data masking for PII
✅ Governance documentation

### Week 3 Deliverables:
✅ DABs project structure
✅ Multi-environment configuration (dev/uat/prod)
✅ Automated deployment pipeline
✅ Production-ready data platform
✅ Monitoring and alerting
✅ Complete documentation and runbooks

**Total: 3 weeks, 90 hours, Production-ready Enterprise Data Platform!**

## Data Governance & Security Implementation Guide

---

### 1. Row-Level Security (RLS) Implementation

#### Use Case: Restrict data access based on user roles
Sales representatives should only see their own region's data.

#### Implementation Pattern:
```sql
-- Create a secured view with row-level filtering
CREATE OR REPLACE VIEW dev.gold.sales_secure AS
SELECT 
  sale_id,
  customer_id,
  product_id,
  quantity,
  sale_amount,
  sale_date,
  region
FROM dev.gold.sales_summary
WHERE 
  -- Admin group sees all data
  is_account_group_member('data_engineers') 
  OR 
  -- Sales reps see only their region
  (is_account_group_member('sales_us') AND region = 'US')
  OR
  (is_account_group_member('sales_eu') AND region = 'EU')
  OR
  (is_account_group_member('sales_apac') AND region = 'APAC');

-- Grant access to the secured view
GRANT SELECT ON VIEW dev.gold.sales_secure TO `analysts`;
```

#### Alternative: User-specific filtering
```sql
CREATE OR REPLACE VIEW dev.gold.customers_by_owner AS
SELECT *
FROM dev.silver.customers_clean
WHERE 
  account_owner = current_user() 
  OR is_account_group_member('executives');
```

---

### 2. Column-Level Security (CLS) & Dynamic Data Masking

#### Use Case: Protect PII while allowing analytics

#### Email Masking:
```sql
CREATE OR REPLACE VIEW dev.gold.customers_masked AS
SELECT 
  customer_id,
  name,
  -- Full email for data engineers, masked for others
  CASE 
    WHEN is_account_group_member('data_engineers') THEN email
    ELSE CONCAT('***@', SPLIT(email, '@')[1])
  END AS email,
  city,
  state,
  -- Full phone for executives, masked for analysts
  CASE 
    WHEN is_account_group_member('executives') THEN phone
    WHEN is_account_group_member('data_engineers') THEN phone
    ELSE CONCAT('***-***-', RIGHT(phone, 4))
  END AS phone,
  signup_date
FROM dev.silver.customers_clean;
```

#### Advanced Masking Patterns:
```sql
-- Credit Card Masking
CASE 
  WHEN is_account_group_member('finance') THEN credit_card
  ELSE CONCAT('****-****-****-', RIGHT(credit_card, 4))
END AS credit_card_number

-- SSN Masking  
CASE 
  WHEN is_account_group_member('hr_admin') THEN ssn
  ELSE CONCAT('***-**-', RIGHT(ssn, 4))
END AS ssn

-- Salary Banding (instead of exact value)
CASE 
  WHEN is_account_group_member('executives') THEN salary
  ELSE 
    CASE 
      WHEN salary < 50000 THEN '<$50K'
      WHEN salary < 100000 THEN '$50K-$100K'
      WHEN salary < 150000 THEN '$100K-$150K'
      ELSE '$150K+'
    END
END AS salary_band
```

---

### 3. Unity Catalog Governance Features

#### Table & Column Comments (Documentation)
```sql
-- Add table comments
COMMENT ON TABLE dev.gold.customer_metrics IS 
  'Customer analytics and KPIs. Refreshed daily at 6 AM UTC. 
   Owner: Data Engineering Team. Contains PII - restricted access.';

-- Add column comments
ALTER TABLE dev.gold.customer_metrics 
ALTER COLUMN customer_id COMMENT 'Unique customer identifier';

ALTER TABLE dev.gold.customer_metrics 
ALTER COLUMN lifetime_value COMMENT 'Total revenue from customer (USD)';

ALTER TABLE dev.gold.customer_metrics 
ALTER COLUMN email COMMENT 'Customer email - PII - masked for non-admin users';
```

#### Unity Catalog Tags
```sql
-- Create tags for data classification
CREATE TAG IF NOT EXISTS dev.pii_level;
CREATE TAG IF NOT EXISTS dev.data_classification;

-- Apply tags to tables
ALTER TABLE dev.silver.customers_clean 
SET TAGS ('pii_level' = 'high', 'data_classification' = 'confidential');

ALTER TABLE dev.gold.sales_summary 
SET TAGS ('pii_level' = 'none', 'data_classification' = 'internal');

-- Apply tags to columns
ALTER TABLE dev.silver.customers_clean 
ALTER COLUMN email SET TAGS ('pii_level' = 'high');

ALTER TABLE dev.silver.customers_clean 
ALTER COLUMN phone SET TAGS ('pii_level' = 'high');
```

---

### 4. Permission Management

#### Create User Groups:
```sql
-- Run these in Databricks Account Console or via API
-- Groups: data_engineers, analysts, executives, sales_us, sales_eu
```

#### Grant Permissions:
```sql
-- Schema-level permissions
GRANT USAGE ON SCHEMA dev.bronze TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.silver TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.gold TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.gold TO `analysts`;  -- Read-only for analysts

-- Table-level permissions
GRANT SELECT ON TABLE dev.gold.customer_metrics TO `analysts`;
GRANT SELECT ON TABLE dev.gold.sales_summary TO `analysts`;
GRANT ALL PRIVILEGES ON TABLE dev.bronze.customers_raw TO `data_engineers`;

-- Secured view access
GRANT SELECT ON VIEW dev.gold.customers_masked TO `analysts`;
GRANT SELECT ON VIEW dev.gold.sales_secure TO `sales_us`;
GRANT SELECT ON VIEW dev.gold.sales_secure TO `sales_eu`;

-- Revoke direct access to sensitive tables
REVOKE SELECT ON TABLE dev.silver.customers_clean FROM `analysts`;
```

---

### 5. Audit Logging & Monitoring

#### Query Audit Logs:
```sql
-- Check who accessed sensitive tables
SELECT 
  event_date,
  user_identity.email,
  request_params.full_name_arg AS table_name,
  action_name
FROM system.access.audit
WHERE 
  action_name = 'getTable'
  AND request_params.full_name_arg LIKE '%customers%'
  AND event_date >= current_date() - INTERVAL 7 DAYS
ORDER BY event_date DESC;

-- Monitor failed access attempts
SELECT 
  event_date,
  user_identity.email,
  request_params.full_name_arg,
  response.status_code,
  response.error_message
FROM system.access.audit
WHERE 
  response.status_code >= 400
  AND event_date >= current_date() - INTERVAL 1 DAYS;
```

---

### 6. Data Lineage Documentation

#### Track data flow:
```
Source Systems
    |
    v
Bronze Layer (Raw)
    ├── customers_raw
    ├── products_raw
    ├── sales_raw
    └── suppliers_raw
    |
    v
Silver Layer (Cleaned & CDC)
    ├── customers_clean (SCD Type 1)
    ├── products_scd2 (SCD Type 2)
    ├── sales_clean
    └── suppliers_clean
    |
    v
Gold Layer (Aggregated)
    ├── sales_summary
    ├── customer_metrics
    ├── product_analytics
    └── daily_kpis
    |
    v
Consumption Layer
    ├── Databricks Dashboards
    ├── Genie Data Rooms
    └── Secured Views (RLS/CLS/Masking)
```

#### Document in Unity Catalog:
```sql
-- Add lineage information in table comments
COMMENT ON TABLE dev.silver.customers_clean IS 
  'Source: dev.bronze.customers_raw | 
   Transformations: Deduplication, NULL handling, SCD Type 1 | 
   Refresh: Every 1 hour | 
   Dependencies: Bronze layer | 
   Downstream: dev.gold.customer_metrics, dashboards';
```

---

## Governance Checklist

### Week 2 - Day 8-10 Deliverables:
- [ ] Row-level security views created for sales and customers
- [ ] Column-level masking implemented for PII (email, phone)
- [ ] User groups created (data_engineers, analysts, executives)
- [ ] Permissions granted using least privilege model
- [ ] Unity Catalog tags applied (pii_level, data_classification)
- [ ] Table and column comments added for all Gold tables
- [ ] Audit logging queries documented
- [ ] Data lineage documented
- [ ] Security policies tested with different user roles
- [ ] Governance documentation completed

## Declarative Automation Bundles (DABs) Implementation Guide

---

### What are DABs?
Declarative Automation Bundles (DABs) enable Infrastructure-as-Code for Databricks:
- **Version Control**: All configurations in Git
- **Multi-Environment**: Deploy same code to dev/uat/prod with different configs
- **Repeatability**: Consistent deployments across environments
- **CI/CD Integration**: Automated testing and deployment pipelines

---

### 1. Project Structure

```
data-engineering-project/
├── databricks.yml              # Main bundle configuration
├── src/                         # Source code
│   ├── bronze/
│   │   ├── load_customers.py
│   │   ├── load_products.py
│   │   └── load_sales.py
│   ├── silver/
│   │   ├── scd_type1_customers.sql
│   │   └── scd_type2_products.sql
│   ├── gold/
│   │   ├── sales_summary.sql
│   │   ├── customer_metrics.sql
│   │   └── product_analytics.sql
│   └── governance/
│       ├── create_secured_views.sql
│       └── apply_permissions.sql
├── resources/                   # Resource definitions
│   ├── jobs/
│   │   ├── bronze_ingestion_job.yml
│   │   ├── silver_cdc_job.yml
│   │   └── gold_aggregation_job.yml
│   └── pipelines/
│       └── medallion_pipeline.yml
├── tests/                       # Unit and integration tests
│   ├── test_bronze_load.py
│   ├── test_silver_transforms.py
│   └── test_gold_aggregates.py
└── README.md
```

---

### 2. Main Configuration File: databricks.yml

```yaml
bundle:
  name: data_engineering_project

include:
  - resources/*.yml

variables:
  catalog:
    description: Unity Catalog name
    default: dev
  
  warehouse_id:
    description: SQL Warehouse ID
    default: "your_warehouse_id"

# Default workspace settings
workspace:
  host: https://your-workspace.cloud.databricks.com

artifacts:
  default:
    type: whl
    build: poetry build
    path: .

# Multi-environment targets
targets:
  # Development Environment
  dev:
    mode: development
    default: true
    workspace:
      host: https://your-workspace.cloud.databricks.com
      root_path: /Workspace/Users/${workspace.current_user.userName}/.bundle/dev
    
    variables:
      catalog: dev
      warehouse_id: "dev_warehouse_id"
      cluster_id: "dev_cluster_id"
    
    resources:
      jobs:
        bronze_ingestion:
          name: "[DEV] Bronze Ingestion"
          schedule:
            quartz_cron_expression: "0 0 * * * ?"  # Every hour
            timezone_id: "UTC"
        
        silver_cdc:
          name: "[DEV] Silver CDC Processing"
          schedule:
            quartz_cron_expression: "0 30 * * * ?"  # 30 minutes after bronze
        
        gold_aggregation:
          name: "[DEV] Gold Aggregation"
          schedule:
            quartz_cron_expression: "0 0 2 * * ?"  # 2 AM daily

  # UAT Environment
  uat:
    mode: development
    workspace:
      host: https://your-workspace.cloud.databricks.com
      root_path: /Workspace/Shared/.bundle/uat
    
    variables:
      catalog: uat
      warehouse_id: "uat_warehouse_id"
      cluster_id: "uat_cluster_id"
    
    resources:
      jobs:
        bronze_ingestion:
          name: "[UAT] Bronze Ingestion"
          schedule:
            quartz_cron_expression: "0 0 */2 * * ?"  # Every 2 hours
        
        silver_cdc:
          name: "[UAT] Silver CDC Processing"
          schedule:
            quartz_cron_expression: "0 30 */2 * * ?"  # 30 min after bronze
        
        gold_aggregation:
          name: "[UAT] Gold Aggregation"
          schedule:
            quartz_cron_expression: "0 0 3 * * ?"  # 3 AM daily

  # Production Environment
  prod:
    mode: production
    workspace:
      host: https://your-workspace.cloud.databricks.com
      root_path: /Workspace/Shared/.bundle/prod
    
    variables:
      catalog: prod
      warehouse_id: "prod_warehouse_id"
      cluster_id: "prod_cluster_id"
    
    # Production has stricter permissions
    permissions:
      - level: CAN_VIEW
        group_name: analysts
      - level: CAN_MANAGE
        group_name: data_engineers
    
    resources:
      jobs:
        bronze_ingestion:
          name: "[PROD] Bronze Ingestion"
          schedule:
            quartz_cron_expression: "0 0 * * * ?"  # Every hour
          email_notifications:
            on_failure:
              - data-engineering-team@company.com
        
        silver_cdc:
          name: "[PROD] Silver CDC Processing"
          schedule:
            quartz_cron_expression: "0 30 * * * ?"  # 30 min after bronze
          email_notifications:
            on_failure:
              - data-engineering-team@company.com
        
        gold_aggregation:
          name: "[PROD] Gold Aggregation"
          schedule:
            quartz_cron_expression: "0 0 1 * * ?"  # 1 AM daily
          email_notifications:
            on_failure:
              - data-engineering-team@company.com
            on_success:
              - data-engineering-team@company.com
          
          # Retry configuration for production
          max_retries: 3
          retry_on_timeout: true
```

---

### 3. Job Resource Definition Example

**File: resources/jobs/bronze_ingestion_job.yml**
```yaml
resources:
  jobs:
    bronze_ingestion:
      name: "Bronze Ingestion Job"
      
      tasks:
        - task_key: load_customers
          notebook_task:
            notebook_path: ../src/bronze/load_customers.py
            base_parameters:
              catalog: ${var.catalog}
              environment: ${bundle.target}
          existing_cluster_id: ${var.cluster_id}
        
        - task_key: load_products
          depends_on:
            - task_key: load_customers
          notebook_task:
            notebook_path: ../src/bronze/load_products.py
            base_parameters:
              catalog: ${var.catalog}
          existing_cluster_id: ${var.cluster_id}
        
        - task_key: load_sales
          depends_on:
            - task_key: load_customers
            - task_key: load_products
          notebook_task:
            notebook_path: ../src/bronze/load_sales.py
            base_parameters:
              catalog: ${var.catalog}
          existing_cluster_id: ${var.cluster_id}
        
        - task_key: data_quality_check
          depends_on:
            - task_key: load_customers
            - task_key: load_products
            - task_key: load_sales
          sql_task:
            warehouse_id: ${var.warehouse_id}
            query:
              query: |
                SELECT 
                  COUNT(*) as customer_count,
                  SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) as null_emails
                FROM ${var.catalog}.bronze.customers_raw;
      
      # Job-level settings
      max_concurrent_runs: 1
      timeout_seconds: 3600
      
      tags:
        environment: ${bundle.target}
        layer: bronze
        team: data-engineering
```

---

### 4. Pipeline Resource Definition

**File: resources/pipelines/medallion_pipeline.yml**
```yaml
resources:
  pipelines:
    medallion_etl:
      name: "Medallion Architecture Pipeline"
      target: ${var.catalog}
      
      libraries:
        - notebook:
            path: ../src/bronze/load_customers.py
        - notebook:
            path: ../src/silver/scd_type1_customers.sql
        - notebook:
            path: ../src/gold/customer_metrics.sql
      
      configuration:
        catalog: ${var.catalog}
        environment: ${bundle.target}
      
      clusters:
        - label: default
          autoscale:
            min_workers: 1
            max_workers: 5
            mode: ENHANCED
      
      continuous: false
      photon: true
      
      development: ${bundle.target == "dev"}
```

---

### 5. DABs CLI Commands

#### Initialize new project:
```bash
databricks bundle init
```

#### Validate configuration:
```bash
# Validate dev environment
databricks bundle validate -t dev

# Validate UAT environment  
databricks bundle validate -t uat

# Validate prod environment
databricks bundle validate -t prod
```

#### Deploy to environments:
```bash
# Deploy to dev (default)
databricks bundle deploy

# Deploy to UAT
databricks bundle deploy -t uat

# Deploy to prod
databricks bundle deploy -t prod
```

#### Run jobs after deployment:
```bash
# Run bronze ingestion in dev
databricks bundle run bronze_ingestion -t dev

# Run in prod
databricks bundle run bronze_ingestion -t prod
```

#### Destroy resources:
```bash
# Remove dev deployment
databricks bundle destroy -t dev

# Remove UAT deployment
databricks bundle destroy -t uat
```

---

### 6. CI/CD Integration (GitHub Actions Example)

**File: .github/workflows/deploy.yml**
```yaml
name: Deploy Data Engineering Pipeline

on:
  push:
    branches:
      - main        # Deploy to prod on main
      - develop     # Deploy to dev on develop
      - uat         # Deploy to UAT on uat branch

jobs:
  deploy:
    runs-on: ubuntu-latest
    
    steps:
      - name: Checkout code
        uses: actions/checkout@v3
      
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      
      - name: Install Databricks CLI
        run: |
          pip install databricks-cli
      
      - name: Determine target environment
        id: env
        run: |
          if [[ "${{ github.ref }}" == "refs/heads/main" ]]; then
            echo "target=prod" >> $GITHUB_OUTPUT
          elif [[ "${{ github.ref }}" == "refs/heads/uat" ]]; then
            echo "target=uat" >> $GITHUB_OUTPUT
          else
            echo "target=dev" >> $GITHUB_OUTPUT
          fi
      
      - name: Configure Databricks authentication
        run: |
          echo "DATABRICKS_HOST=${{ secrets.DATABRICKS_HOST }}" >> $GITHUB_ENV
          echo "DATABRICKS_TOKEN=${{ secrets.DATABRICKS_TOKEN }}" >> $GITHUB_ENV
      
      - name: Validate bundle
        run: |
          databricks bundle validate -t ${{ steps.env.outputs.target }}
      
      - name: Deploy to Databricks
        run: |
          databricks bundle deploy -t ${{ steps.env.outputs.target }}
      
      - name: Run tests (dev only)
        if: steps.env.outputs.target == 'dev'
        run: |
          databricks bundle run bronze_ingestion -t dev
          # Run validation queries
```

---

### 7. Environment-Specific Configuration

| Setting | DEV | UAT | PROD |
|---------|-----|-----|------|
| **Catalog** | dev | uat | prod |
| **Schedule** | Hourly | Every 2 hours | Hourly |
| **Compute** | Single node | Small cluster | Autoscaling cluster |
| **Retries** | 0 | 1 | 3 |
| **Notifications** | None | On failure | On failure & success |
| **Data Retention** | 7 days | 30 days | 365 days |
| **Permissions** | Engineers only | Engineers + QA | Engineers + Analysts |

---

### 8. Best Practices

#### Configuration Management:
✅ **Use variables** for environment-specific values (catalog names, cluster IDs)
✅ **Store secrets** in Databricks secrets scope, not in databricks.yml
✅ **Version control** all configuration files
✅ **Document** all parameters and their purposes

#### Deployment Strategy:
✅ **Always deploy to dev first** and test thoroughly
✅ **Deploy to UAT** for business user acceptance testing
✅ **Deploy to prod** only after UAT approval
✅ **Use Git branches** to manage deployments (main=prod, develop=dev, uat=uat)

#### Testing:
✅ **Validate before deploy** using `databricks bundle validate`
✅ **Run integration tests** in dev after deployment
✅ **Monitor job runs** after production deployment
✅ **Set up alerts** for failed jobs

#### Rollback Strategy:
✅ **Tag releases** in Git for easy rollback
✅ **Keep previous bundle version** deployed
✅ **Test rollback procedure** in dev environment
✅ **Document rollback steps** in runbook

---

## DABs Implementation Checklist

### Week 3 - Day 11-15 Deliverables:
- [ ] Project structure created with src/, resources/, tests/ folders
- [ ] databricks.yml configured with dev/uat/prod targets
- [ ] Job definitions created for bronze, silver, gold layers
- [ ] Environment-specific variables configured
- [ ] Cluster and warehouse IDs set for each environment
- [ ] Bundle validated for all three environments
- [ ] Deployed successfully to dev environment
- [ ] Deployed successfully to UAT environment
- [ ] Deployed successfully to prod environment
- [ ] CI/CD pipeline configured (GitHub Actions / Azure DevOps)
- [ ] Email notifications set up for production jobs
- [ ] Permissions configured for each environment
- [ ] Rollback procedure documented and tested
- [ ] Team trained on deployment process

## Code Templates & References

### Bronze Data Generation Template (Python)
```python
import random
from datetime import datetime, timedelta

# Generate customers with nulls and duplicates
customers_data = [
    (1, 'John Doe', 'john@email.com', 'New York', 'NY', '2024-01-15', '555-0101'),
    (2, 'Jane Smith', None, 'Los Angeles', 'CA', '2024-02-20', '555-0102'),  # NULL email
    (1, 'John Doe', 'john@email.com', 'New York', 'NY', '2024-01-15', '555-0101'),  # Duplicate
    # ... generate 1000+ records
]

df_customers = spark.createDataFrame(customers_data, 
    ['customer_id', 'name', 'email', 'city', 'state', 'signup_date', 'phone'])
df_customers.write.mode('overwrite').saveAsTable('dev.bronze.customers_raw')
```

### COPY INTO Template
```sql
COPY INTO dev.bronze.customers_raw
FROM 's3://your-bucket/customers/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

### SCD Type 1 Template
```sql
MERGE INTO dev.silver.customers_clean AS target
USING (
  SELECT customer_id, name, email, city, state, signup_date, phone
  FROM dev.bronze.customers_raw
  WHERE email IS NOT NULL  -- Data quality filter
  QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY signup_date DESC) = 1
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

### SCD Type 2 Template
```sql
-- Step 1: Close expired records
MERGE INTO dev.silver.products_scd2 AS t
USING (
  SELECT product_id, product_name, category, price
  FROM dev.bronze.products_raw
  QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY product_id) = 1
) AS s
ON t.product_id = s.product_id AND t.is_current = true
WHEN MATCHED AND (t.product_name != s.product_name OR t.price != s.price) THEN
  UPDATE SET 
    t.is_current = false,
    t.end_date = current_date();

-- Step 2: Insert new versions
INSERT INTO dev.silver.products_scd2
SELECT 
  s.product_id, s.product_name, s.category, s.price,
  current_date() AS effective_date,
  NULL AS end_date,
  true AS is_current
FROM source s
INNER JOIN dev.silver.products_scd2 t
  ON s.product_id = t.product_id AND t.end_date = current_date();
```

### Window Functions Examples
```sql
-- Ranking
SELECT 
  product_name,
  SUM(sale_amount) as revenue,
  ROW_NUMBER() OVER (ORDER BY SUM(sale_amount) DESC) as rank,
  PERCENT_RANK() OVER (ORDER BY SUM(sale_amount)) as percentile
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY product_name;

-- Running totals
SELECT 
  sale_date,
  sale_amount,
  SUM(sale_amount) OVER (ORDER BY sale_date) as running_total
FROM sales;

-- Moving average
SELECT
  sale_date,
  AVG(sale_amount) OVER (
    ORDER BY sale_date 
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ) as moving_avg_7day
FROM daily_sales;
```

## Dashboard & Genie Setup Guide

### Creating Databricks Dashboard

#### Step-by-Step:
1. **Create New Dashboard**
   - Click "Create" → "Dashboard" in Databricks workspace
   - Name it: "Sales & Customer Analytics Dashboard"

2. **Add Visualizations from Gold Tables**
   ```sql
   -- Chart 1: Revenue by Region (Bar Chart)
   SELECT region, SUM(total_revenue) as revenue
   FROM dev.gold.sales_summary
   GROUP BY region
   ORDER BY revenue DESC;
   
   -- Chart 2: Top 10 Customers (Table)
   SELECT customer_name, total_purchases, lifetime_value
   FROM dev.gold.customer_metrics
   ORDER BY lifetime_value DESC
   LIMIT 10;
   
   -- Chart 3: Sales Trend (Line Chart)
   SELECT sale_date, daily_revenue, 
          AVG(daily_revenue) OVER (ORDER BY sale_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as moving_avg_7day
   FROM dev.gold.daily_kpis
   ORDER BY sale_date;
   
   -- Chart 4: Product Category Performance (Pie Chart)
   SELECT category, SUM(sales_amount) as total_sales
   FROM dev.gold.product_analytics
   GROUP BY category;
   
   -- Chart 5: Price History (Line Chart - using SCD Type 2)
   SELECT effective_date, product_name, price
   FROM dev.silver.products_scd2
   WHERE product_id = 1
   ORDER BY effective_date;
   ```

3. **Add Dashboard Filters**
   - Date range filter (start_date, end_date)
   - Region dropdown filter
   - Product category multiselect filter

4. **Configure Auto-Refresh**
   - Set refresh schedule (e.g., every 1 hour)
   - Enable background refresh

5. **Share Dashboard**
   - Click "Share" button
   - Add users/groups with view or edit permissions
   - Generate public link if needed

---

### Setting Up Genie Data Room

#### Step-by-Step:
1. **Create Genie Space**
   - Navigate to Genie in Databricks workspace
   - Click "Create Space"
   - Name it: "Sales Analytics Intelligence"

2. **Add Tables to Genie Space**
   - Select tables to include:
     - `dev.gold.sales_summary`
     - `dev.gold.customer_metrics`
     - `dev.gold.product_analytics`
     - `dev.gold.daily_kpis`
   - Genie will automatically understand table schemas and relationships

3. **Test Natural Language Queries**
   ```
   Example queries to try:
   
   1. "Show me the top 10 customers by revenue"
   2. "What's the total sales by region for this year?"
   3. "Which product category has the highest profit margin?"
   4. "Compare sales between Q1 and Q2"
   5. "Show me customers who haven't purchased in the last 90 days"
   6. "What's the average order value by customer segment?"
   7. "Which products have had price increases? Show me the history"
   8. "Calculate the 7-day moving average of daily sales"
   9. "Show me year-over-year revenue growth by region"
   10. "What percentage of total revenue comes from the top 20% of customers?"
   ```

4. **Genie Benefits**
   - ✅ No SQL knowledge required
   - ✅ Automatic chart generation
   - ✅ Conversational follow-up questions
   - ✅ Data exploration for non-technical users
   - ✅ Built-in data governance (uses Unity Catalog permissions)

5. **Share Genie Space**
   - Add users to the space
   - Set permissions (can query, can edit)
   - Users can save their favorite queries

---

### Dashboard vs Genie: When to Use

| Feature | Databricks Dashboard | Genie Data Room |
|---------|---------------------|------------------|
| **Best For** | Curated KPIs and metrics | Ad-hoc exploration |
| **User Type** | Executives, managers | Analysts, business users |
| **Query Method** | Pre-built SQL visualizations | Natural language |
| **Customization** | High (manual chart config) | Auto-generated |
| **Use Case** | Daily monitoring, reports | Investigation, what-if analysis |
| **Update Frequency** | Scheduled refresh | Real-time queries |

**Recommendation**: Use both!
- Dashboard for regular monitoring and reporting
- Genie for ad-hoc questions and deep dives

## Unity Catalog Structure

### Recommended Schema Organization
```
dev                              (catalog)
├── bronze                       (schema - raw ingested data)
│   ├── customers_raw
│   ├── products_raw  
│   ├── sales_raw
│   └── suppliers_raw
│
├── silver                       (schema - cleaned & transformed)
│   ├── customers_clean          (SCD Type 1)
│   ├── products_scd2            (SCD Type 2 with history)
│   ├── sales_clean
│   └── suppliers_clean
│
└── gold                         (schema - aggregated analytics)
    ├── sales_summary
    ├── customer_metrics
    ├── product_analytics
    └── daily_kpis
```

### Setup Commands
```sql
-- Create schemas if they don't exist
CREATE SCHEMA IF NOT EXISTS dev.bronze;
CREATE SCHEMA IF NOT EXISTS dev.silver;
CREATE SCHEMA IF NOT EXISTS dev.gold;
```

---

## Best Practices & Tips

### Data Quality in Bronze
✅ **DO** include nulls and duplicates in Bronze (represents real-world data)
✅ **DO** keep all source data, even invalid records
✅ **DO** add ingestion timestamp for tracking

### CDC Implementation
✅ **SCD Type 1**: Use when only current state matters (customer contact info)
✅ **SCD Type 2**: Use when history is critical (product prices, terms changes)
✅ **Always deduplicate source** before MERGE to avoid "multiple matching rows" error

### Performance Optimization
✅ **Partition large tables** by date columns (e.g., `sale_date`)
✅ **Z-ORDER** on commonly filtered columns (e.g., `customer_id`, `product_id`)
✅ **Use QUALIFY** instead of subqueries for window function filtering
✅ **Materialize** intermediate results for complex transformations

### Window Function Tips
- `ROW_NUMBER()`: Assign unique sequential numbers (good for deduplication)
- `RANK()`: Assigns same rank to ties, skips next rank
- `DENSE_RANK()`: Assigns same rank to ties, doesn't skip
- `LAG()/LEAD()`: Access previous/next row values
- `FIRST_VALUE()/LAST_VALUE()`: Get first/last value in window
- Use `QUALIFY` to filter on window function results directly

---

## Validation Queries

### Check for duplicates
```sql
SELECT customer_id, COUNT(*) as cnt
FROM dev.bronze.customers_raw
GROUP BY customer_id
HAVING COUNT(*) > 1;
```

### Verify SCD Type 2 history
```sql
SELECT product_id, COUNT(*) as versions,
       SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_count
FROM dev.silver.products_scd2
GROUP BY product_id
HAVING current_count != 1;  -- Should return 0 rows
```

### Check referential integrity
```sql
SELECT COUNT(*) as orphan_sales
FROM dev.silver.sales_clean s
LEFT JOIN dev.silver.customers_clean c ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
```

## Comprehensive Checklist - 3 Week Sprint (90 Hours)

---

# WEEK 1: Core Pipeline (30 Hours)

## Day 1-2: Setup & Bronze Layer (12 hours)
### Monday
- [ ] Create multi-environment catalog structure:
  - [ ] dev.bronze, dev.silver, dev.gold
  - [ ] uat.bronze, uat.silver, uat.gold
  - [ ] prod.bronze, prod.silver, prod.gold
- [ ] Generate sample data scripts:
  - [ ] Customers (1000 rows, with nulls/duplicates)
  - [ ] Products (500 rows, with nulls/duplicates)
  - [ ] Sales (5000 rows, with nulls)
  - [ ] Suppliers (100 rows)
- [ ] Load data to dev.bronze tables
- [ ] Validate nulls and duplicates exist

### Tuesday
- [ ] Implement ingestion layer (COPY INTO or Lakeflow Pipeline)
- [ ] Export Bronze data to cloud storage (if using COPY INTO)
- [ ] Test ingestion with sample data
- [ ] Build SCD Type 1 for customers
- [ ] Create dev.silver.customers_clean table
- [ ] Write MERGE statement with deduplication
- [ ] Test and validate customer data

## Day 3: Silver - SCD Type 2 (6 hours)
### Wednesday
- [ ] Create dev.silver.products_scd2 table structure
  - [ ] Add effective_date, end_date, is_current, version columns
- [ ] Write MERGE statement (close expired records)
- [ ] Write INSERT statement (add new versions)
- [ ] Test with product price changes
- [ ] Verify history preservation works correctly

## Day 4-5: Gold Layer (12 hours)
### Thursday
- [ ] Create Gold aggregate tables:
  - [ ] dev.gold.sales_summary
  - [ ] dev.gold.customer_metrics
  - [ ] dev.gold.product_analytics
  - [ ] dev.gold.daily_kpis
- [ ] Build KPI queries with window functions:
  - [ ] ROW_NUMBER(), RANK(), DENSE_RANK()
  - [ ] LAG(), LEAD(), FIRST_VALUE()
  - [ ] Moving averages and running totals

### Friday
- [ ] Implement 4-way joins (sales → customers → products → suppliers)
- [ ] Create self-joins for comparison analysis
- [ ] Run data quality validation across all layers
- [ ] Document any issues found

---

# WEEK 2: Visualization, Intelligence & Governance (30 Hours)

## Day 6-7: Dashboards & Genie (12 hours)
### Monday
- [ ] Create Databricks Dashboard:
  - [ ] Sales Performance charts (revenue trends, regional analysis)
  - [ ] Customer Analytics visualizations (CLV, segmentation)
  - [ ] Product Insights (best-sellers, price history)
- [ ] Add dashboard features:
  - [ ] Date range filters
  - [ ] Region/category filters
  - [ ] Drill-down capabilities
  - [ ] Auto-refresh schedule

### Tuesday
- [ ] Set up Genie Data Room
- [ ] Add Gold tables to Genie space
- [ ] Configure natural language query interface
- [ ] Test common business questions:
  - [ ] "Top 10 customers by revenue"
  - [ ] "Average order value by region"
  - [ ] "Highest profit margin products"
  - [ ] "Q1 vs Q2 sales comparison"
- [ ] Create business user training guide

## Day 8-10: Data Governance & Security (18 hours)
### Wednesday
- [ ] Unity Catalog governance:
  - [ ] Add table/column comments
  - [ ] Document data lineage
  - [ ] Create governance tags (pii_level, data_classification)
- [ ] Implement Row-Level Security:
  - [ ] Create row filters using current_user()
  - [ ] Apply filters to customers and sales tables
  - [ ] Test with different user roles
- [ ] Implement Column-Level Security:
  - [ ] Create column masks for email and phone
  - [ ] Use is_account_group_member() for role-based access
  - [ ] Test masking functionality

### Thursday
- [ ] Implement Dynamic Data Masking:
  - [ ] Email masking (***@domain.com)
  - [ ] Phone masking (***-***-1234)
  - [ ] Credit card masking (****-****-****-1234)
- [ ] Create secured views:
  - [ ] dev.gold.customers_secure_view
  - [ ] dev.gold.sales_secure_view
- [ ] Test masking with different user groups

### Friday
- [ ] Fine-grained access control:
  - [ ] Create user groups (data_engineers, analysts, executives)
  - [ ] Grant schema permissions (USAGE, SELECT, MODIFY)
  - [ ] Grant table permissions
  - [ ] Implement least privilege access
- [ ] Governance documentation:
  - [ ] Document all security policies
  - [ ] Set up audit logging
  - [ ] Create compliance reports
  - [ ] Test audit log queries

---

# WEEK 3: SDLC with DABs & Production (30 Hours)

## Day 11-12: DABs Setup (12 hours)
### Monday
- [ ] Initialize DAB project:
  - [ ] Run `databricks bundle init`
  - [ ] Review folder structure (src/, resources/, tests/)
- [ ] Configure databricks.yml:
  - [ ] Define dev/uat/prod targets
  - [ ] Set environment variables (catalog, clusters, warehouses)
  - [ ] Configure resource paths
  - [ ] Set deployment settings
- [ ] Organize code into bundle structure:
  - [ ] Move notebooks to src/ folder
  - [ ] Create bronze/, silver/, gold/, governance/ folders
  - [ ] Create reusable modules

### Tuesday
- [ ] Define bundle resources:
  - [ ] Create job definitions (bronze_ingestion_job.yml)
  - [ ] Create job definitions (silver_cdc_job.yml)
  - [ ] Create job definitions (gold_aggregation_job.yml)
  - [ ] Configure cluster policies
  - [ ] Set up task dependencies
- [ ] CI/CD integration:
  - [ ] Configure Git integration
  - [ ] Create GitHub Actions / Azure DevOps workflow
  - [ ] Document deployment process

## Day 13-14: Multi-Environment Deployment (12 hours)
### Wednesday
- [ ] Deploy to DEV:
  - [ ] Run `databricks bundle validate -t dev`
  - [ ] Run `databricks bundle deploy -t dev`
  - [ ] Verify all resources created
- [ ] Test in DEV:
  - [ ] Run Bronze → Silver → Gold pipeline
  - [ ] Validate data transformations
  - [ ] Test security policies
  - [ ] Verify dashboard and Genie work
- [ ] Deploy to UAT:
  - [ ] Run `databricks bundle validate -t uat`
  - [ ] Run `databricks bundle deploy -t uat`
  - [ ] Load UAT test data

### Thursday
- [ ] UAT Testing:
  - [ ] Business user acceptance testing
  - [ ] Dashboard testing
  - [ ] Genie testing
  - [ ] Security validation
  - [ ] Performance testing
- [ ] Bug fixes:
  - [ ] Address issues found
  - [ ] Update documentation
  - [ ] Re-test affected areas
- [ ] Production preparation:
  - [ ] Create deployment checklist
  - [ ] Schedule production deployment window
  - [ ] Notify stakeholders

## Day 15: Production Deployment (6 hours)
### Friday
- [ ] Production deployment:
  - [ ] Run `databricks bundle validate -t prod`
  - [ ] Run `databricks bundle deploy -t prod`
  - [ ] Verify production resources
- [ ] Configure production workflows:
  - [ ] Schedule Bronze ingestion (hourly/daily)
  - [ ] Schedule Silver CDC processing
  - [ ] Schedule Gold aggregations
  - [ ] Set up email notifications
  - [ ] Configure retry policies
- [ ] Monitoring & documentation:
  - [ ] Set up monitoring dashboards
  - [ ] Configure alerting rules
  - [ ] Create operational runbook
  - [ ] Document troubleshooting procedures
  - [ ] Prepare training materials
  - [ ] Project retrospective

---

## Final Success Criteria

### Technical Deliverables:
✅ Multi-environment catalogs (dev/uat/prod)
✅ Bronze layer with 4 tables and data quality issues
✅ Silver layer with SCD Type 1 & Type 2
✅ Gold layer with 4+ analytics tables
✅ Window functions (5+ types) implemented
✅ 4-way joins working correctly
✅ Databricks Dashboard with 5+ visualizations
✅ Genie Data Room with natural language queries

### Governance & Security:
✅ Row-level security implemented
✅ Column-level security implemented
✅ Dynamic data masking for PII
✅ User groups and permissions configured
✅ Unity Catalog tags applied
✅ Audit logging configured
✅ Governance documentation complete

### SDLC & Automation:
✅ DABs project structure created
✅ databricks.yml configured for 3 environments
✅ Job definitions for all pipeline stages
✅ CI/CD pipeline integrated
✅ Deployed to dev/uat/prod successfully
✅ Automated workflows scheduled
✅ Monitoring and alerting configured
✅ Rollback procedures documented

---

## Total: 3 Weeks | 90 Hours | Production-Ready Enterprise Data Platform

**Ready to start?** Begin with Week 1, Day 1 - Multi-environment setup!